# yy14 — WNM `exc_vis_manuscript`: full ingestion

**Source CSVs** (`data/wnm_exc_vis_manuscript/`):
| File | Cells | Content |
|------|-------|---------|
| `FullMorphMetaData_Master.csv` | 341 | MET labels, CCF soma coords, cre line, classifier outputs |
| `AxonRawReatureWide.csv` | 345 | 51 axon + apical dendrite morphology features |
| `ProjectionMatrix_tip_and_branch_roll_up.csv` | 345 | 220 brain-region projection columns (152 ipsi, 68 contra) |

**Schema identifiers:**
- `project_id` = `"visp_wnm"`
- `dataset_id` = `"visp_exc_wnm"`

## Delta lake tables written by this notebook

| Path | Schema class | Source |
|------|---|---|
| `cellfeatures/wnm_exc_axon_features` | `CellFeatureMatrix` (wide-form) | `AxonRawReatureWide.csv` |
| `cellfeatureset/wnm_exc_axon` | `CellFeatureSet` | same |
| `cellfeaturedefinition/wnm_exc_axon` | `CellFeatureDefinition` | same |
| `projectionmeasurementmatrix` | `ProjectionMeasurementMatrix` (metadata) | `ProjectionMatrix_*.csv` |
| `projectionmeasurementmatrix/wnm_exc_proj_ipsi` | wide-form matrix | same (152 ipsi cols) |
| `projectionmeasurementmatrix/wnm_exc_proj_contra` | wide-form matrix | same (68 contra cols) |
| `singlecellreconstruction` | `SingleCellReconstruction` | `FullMorphMetaData_Master.csv` |

**Not written (skipped):** `cre_line`, `azimuth`, `altitude`, subclass classifier columns — no schema home yet.  
**Already in delta lake:** `celltoclustermapping` for MET types (written by prior notebook).


## ⚠️ Schema gap: `CellFeatureMatrix` pointer table is missing

### Problem

The schema defines `CellFeatureMatrix` as the linking object between a `CellFeatureSet` (metadata) and the actual wide-form data table (the numbers). It has five fields:

| Field | Purpose |
|---|---|
| `id` | Human-readable identifier for this matrix instance |
| `project_id` | Project scope |
| `feature_set_id` | Reference to the `CellFeatureSet` |
| `parquet_path` | `file://` (or `s3://`) path to the wide-form delta table |
| `cell_index_column` | Which column in the table holds the `DataItem` ID |

Without it, there is no schema-level pointer from a `CellFeatureSet` to its actual data. A consumer reading the schema has no way to discover where the numbers live — they must know the `cellfeatures/*` path convention by convention rather than by lookup.

### Current state

All existing `cellfeatures/*` tables were written without a corresponding `CellFeatureMatrix` row. No `cellfeaturematrix` delta table exists yet in the combined datasets.

### Fix needed

1. Create a `cellfeaturematrix` delta table in the combined datasets
2. Backfill one `CellFeatureMatrix` row per existing `cellfeatures/*` table
3. Add a `CellFeatureMatrix` write step to every future notebook that writes a `cellfeatures/*` table (including this one for `wnm_exc_axon_features`)

### Open design question

`cellfeatures/exc_morph_features` contains rows for two `project_id` values (`visp_exc_patchseq` and `visp_exc_wnm`) in the same physical delta table. Should there be:
- **One** `CellFeatureMatrix` row pointing to the shared table (simpler, but `project_id` is ambiguous), or
- **Two** `CellFeatureMatrix` rows (one per `project_id`) both pointing to the same `parquet_path` (explicit, but redundant path)?


In [1]:
import os
import sys

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

sys.path.append("/root/capsule/src/")
from connects_common_connectivity.arrow_utils import (
    attach_linkml_metadata,
    build_arrow_schema,
    build_cell_feature_matrix_schema,
    models_to_table,
)
from connects_common_connectivity.models import (
    CellFeatureDefinition,
    CellFeatureSet,
    DataItem,
    DataSet,
    Modality,
    ProjectionMeasurementMatrix,
    ProjectionMeasurementType,
    SingleCellReconstruction,
    SpatialLocation,
)


In [2]:
# --- Path constants ---
CSV_ROOT = "/root/capsule/data/wnm_exc_vis_manuscript"

# Combined datasets live in scratch; this is where the new tables will be appended
CODEOCEAN_COMBINED = "/scratch/combined_datasets"
LOCAL_COMBINED = "/scratch/combined_datasets"  # same path; update if needed
COMBINED_ROOT = CODEOCEAN_COMBINED if os.path.exists(CODEOCEAN_COMBINED) else LOCAL_COMBINED

print(f"CSV_ROOT    : {CSV_ROOT}")
print(f"COMBINED_ROOT: {COMBINED_ROOT}")

CSV_ROOT    : /root/capsule/data/wnm_exc_vis_manuscript
COMBINED_ROOT: /scratch/combined_datasets


## Load and inspect source CSVs

In [3]:
# Load the three source CSVs
_wnm_meta = pd.read_csv(os.path.join(CSV_ROOT, "FullMorphMetaData_Master.csv"), index_col=0)
_wnm_axon = pd.read_csv(os.path.join(CSV_ROOT, "AxonRawReatureWide.csv"))
_wnm_proj = pd.read_csv(os.path.join(CSV_ROOT, "ProjectionMatrix_tip_and_branch_roll_up.csv"), index_col=0)

print(f"FullMorphMetaData_Master          : {_wnm_meta.shape}")
print(f"AxonRawReatureWide                : {_wnm_axon.shape}")
print(f"ProjectionMatrix_tip_and_branch   : {_wnm_proj.shape}")

FullMorphMetaData_Master          : (341, 16)
AxonRawReatureWide                : (345, 52)
ProjectionMatrix_tip_and_branch   : (345, 220)


In [4]:
print("=== FullMorphMetaData_Master (head) ===")
display(_wnm_meta.head(3))

=== FullMorphMetaData_Master (head) ===


,predicted_met_type,probability,ccf_soma_location,ccf_soma_location_nolayer,ccf_soma_x,ccf_soma_y,ccf_soma_z,distance_soma_moved_out_of_brain_correction,cre_line,azimuth,altitude,auto_projection_subclass,dend_derived_predicted_subclass,dend_derived_predicted_probability,local_axon_derived_subclass,met_classifier_routing_call
182709_6984-X2452-Y12423_reg.swc,L5 ET-2,0.988,VISpm5,VISpm,8899.823,643.262,4324.473,37.416574,Ai82;Ai139_375886-182709,35.212157,3.519493,ET,ET,0.760219,NaN,ET
182709_7126-X2913-Y10535_reg.swc,L5 ET-3,0.918,VISp5,VISp,9117.453,1064.075,3550.508,24.494897,Ai82;Ai139_375886-182709,48.447344,-0.296991,ET,ET,0.949430,NaN,ET
182724_5937-X3804-Y11955_reg.swc,L5 ET-2,0.724,VISa5,VISa,7168.289,953.689,3959.651,42.426407,Fezf2-CreER;Ai166_405426-182724,41.861469,1.670020,ET,ET,0.928120,NaN,ET


In [5]:
print("=== AxonRawReatureWide (head) ===")
display(_wnm_axon.head(3))

=== AxonRawReatureWide (head) ===


,specimen_id,apical_dendrite_bias_x,apical_dendrite_bias_y,apical_dendrite_depth_pc_0,apical_dendrite_depth_pc_1,apical_dendrite_depth_pc_2,apical_dendrite_depth_pc_3,apical_dendrite_depth_pc_4,apical_dendrite_early_branch_path,apical_dendrite_extent_x,...,axon_max_branch_order,axon_max_euclidean_distance,axon_max_path_distance,axon_mean_contraction,axon_num_branches,axon_soma_percentile_x,axon_soma_percentile_y,axon_total_length,soma_aligned_dist_from_pia,soma_surface_area
0,17109_6201-X4328-Y6753_reg,21.311538,2.306327,-446.959647,-77.960333,-116.411504,-89.641313,-76.897975,0.479785,291.778779,...,18.0,782.489366,2295.547989,0.775924,133.0,0.456671,0.291300,32679.601463,755.648634,0.0
1,17109_6301-X4756-Y24516_reg,39.591795,17.363311,-456.398402,-70.969916,-123.361715,-92.973863,-99.533545,0.484623,255.485239,...,11.0,798.003108,1794.220795,0.789746,83.0,0.434223,0.193180,24944.239274,779.803826,0.0
2,17109_6601-X4384-Y7436_reg,110.472066,-7.655321,-447.486796,-103.720271,-125.564660,-117.194281,-51.706742,0.498860,331.019359,...,10.0,722.175168,2298.330681,0.773319,79.0,0.297082,0.511213,20598.298943,727.831495,0.0


In [6]:
print("=== ProjectionMatrix (head, first 6 cols) ===")
display(_wnm_proj.iloc[:3, :6])

=== ProjectionMatrix (head, first 6 cols) ===


,ipsi_VISam,ipsi_VISp,ipsi_VISpm,ipsi_VISrl,contra_VISpor,ipsi_CP
18864_6734-X4899-Y27447_reg.swc,8287.70664,34450.175934,483.223644,5737.760785,0.000000,0.000000
191812_7938-X6892-Y25312_reg.swc,0.00000,794.102517,0.000000,0.000000,1045.437572,9243.339922
211550_7718-X19461-Y16950_reg.swc,0.00000,6473.751624,0.000000,0.000000,0.000000,0.000000


### Build the intersection of all cell IDs

Strip `.swc` suffix from CSV indices where present; use `specimen_id` for the axon CSV.

In [7]:
# Normalize IDs: strip .swc suffix
meta_ids = {i.replace(".swc", "") for i in _wnm_meta.index}
axon_ids = {str(i) for i in _wnm_axon["specimen_id"]}
proj_ids = {i.replace(".swc", "") for i in _wnm_proj.index}

all_cell_ids = sorted(meta_ids & axon_ids & proj_ids)

print(f"meta_ids : {len(meta_ids)}")
print(f"axon_ids : {len(axon_ids)}")
print(f"proj_ids : {len(proj_ids)}")
print(f"overlap in all 3 : {len(all_cell_ids)}   ← DataItems")

meta_ids : 341
axon_ids : 345
proj_ids : 345
overlap in all 3 : 341   ← DataItems


## Find out what is already NOT in the schemas

### Q: Is `AxonRawReatureWide` features different than existing exc morph wnm features?

**Yes — they are partially overlapping but distinct feature sets.**

Both have 51 feature columns, but only 25 are shared:

| Group | Count | Example cols |
|-------|-------|-------------|
| **Shared** | 25 | All `apical_dendrite_*` basics, `axon_exit_distance/theta`, `soma_aligned_dist_from_pia` |
| **Only in `AxonRawReatureWide`** | 26 | All `axon_*` geometry (length, branches, extent, PCs), `apical_dendrite_frac_*_axon`, `soma_surface_area` |
| **Only in existing `exc_morph_features`** | 26 | All `basal_dendrite_*`, `apical_dendrite_frac_*_basal_dendrite` |

**Interpretation:** `AxonRawReatureWide` contains the WNM-specific **axon geometry features** (the whole point of the dataset), but drops basal dendrite. The existing `exc_morph_features` (patchseq) has basal dendrite but no axon. These should be stored as a **separate feature set** (`wnm_exc_axon_features`), not merged into the existing one.


### Q: Is there a data schema specifically for saving the projection matrix? How does it relate to brain region schema? Does brain region schema have a binary indicator per region?

**Yes, `ProjectionMeasurementMatrix` exists in `projection_schema.yaml`.** It has:
- `region_index` — ordered list of `BrainRegion` references (columns = the 220 ipsi/contra regions)
- `data_item_index` — ordered list of `DataItem` references (rows = 345 cells)
- `values` — a `ZarrArray` holding the matrix values
- `measurement_type`, `modality`, `unit`

**Relationship to `brain_region_schema`:** `region_index` holds `BrainRegion` IDs directly. The 220 projection columns are named like `ipsi_VISp`, `contra_VISpm` — these need parsing (strip `ipsi_`/`contra_` prefix) to match `BrainRegion.acronym` values, and ipsi/contra side should be stored separately or encoded in the `measurement_type`.

**No binary indicator per region in `BrainRegion`.** The `BrainRegion` class is a pure ontology (id, name, parent, children, acronym, color). There is no "has_data_for_dataset" or coverage flag. The closest thing is `BrainRegionAssociation` (DataItem ↔ BrainRegion link), but that is per-cell, not per-dataset.

**Bottom line:** The schema supports this data but the implementation currently expects Zarr backing, which may be over-engineered for a 345×220 matrix. A wide-form delta table is simpler and consistent with how `cellfeatures` are stored — worth discussing.


### Q: Are the MET types in `FullMorphMetaData_Master.csv` the same as in the delta lakes?

**Yes, fully.** All 15 unique `predicted_met_type` values in the CSV are present as cluster IDs in the `visp_met_types` project of the combined delta lake.

The delta lake has 48 total `visp_met_types` clusters (it includes inhibitory MET types + hierarchy roots like `Glutamatergic`, `GABAergic`, `cell`). The 15 excitatory MET types in the CSV are a clean subset:

`L2/3 IT`, `L4 IT`, `L4/L5 IT`, `L5 ET-1 Chrna6`, `L5 ET-2`, `L5 ET-3`, `L5 IT-2`, `L5 NP`, `L5/L6 IT Car3`, `L6 CT-1`, `L6 CT-2`, `L6 IT-1`, `L6 IT-2`, `L6 IT-3`, `L6b`

→ Safe to write `CellToClusterMapping` entries directly using these strings as `target_cluster` IDs under `project_id="visp_met_types"`.


### Q: Are the cre lines in `FullMorphMetaData_Master.csv` already in the schema? Would they fit into `CellGeneData`?

**No, and no — `CellGeneData` is the wrong class.**

The `cre_line` values look like `"Fezf2-CreER;Ai166_439168-191807"` — they encode the **transgenic mouse line + reporter + animal ID**. This is experimental/sample metadata, not gene expression data.

`CellGeneData` / `BarcodingExperimentMetadata` is designed for cell×gene expression matrices from BarSeq/PathSeq/MERFISH experiments (Zarr-backed). Cre lines are not gene expression.

**What fits better:**
- A metadata `CellFeatureSet` with string-type `CellFeatureDefinition` entries — technically valid but semantically off
- A `SingleCellReconstruction`-adjacent metadata table (see Cell 15)
- **No current schema class perfectly covers this.** It may warrant a new `CellMetadata` or `SampleMetadata` class, or simply storing as a wide-form delta table outside the typed schema until a class is defined.

For now the most pragmatic approach is a string-typed feature set or a raw delta table with explicit documentation.


### Q: `SingleCellReconstruction` has `ccf_registered_file` and `soma_location` — these can be extracted from `FullMorphMetaData_Master.csv`

**Correct.** The schema class `SingleCellReconstruction` (`single_cell_schema.yaml`) has:
- `id` → DataItem reference
- `ccf_registered_file` → protocol+path string (e.g. `swc://s3://bucket/cell.swc`)
- `soma_location` → `SpatialLocation` (x, y, z, reference_space)

From `FullMorphMetaData_Master.csv` we can populate:

| Schema field | CSV source | Notes |
|---|---|---|
| `id` | index (e.g. `182709_6984-X2452-Y12423_reg.swc`) stripped of `.swc` | DataItem ID |
| `ccf_registered_file` | index filename | Need to prefix with actual storage path/protocol |
| `soma_location.x` | `ccf_soma_x` | ✓ direct |
| `soma_location.y` | `ccf_soma_y` | ✓ direct |
| `soma_location.z` | `ccf_soma_z` | ✓ direct |
| `soma_location.reference_space` | hardcode `"CCF_v3"` | `ccf_soma_location` is the region name, not the reference space |

`ccf_soma_location` (e.g. `"VISp5"`) is the **region label**, not the reference space — it could feed a `BrainRegionAssociation`, not `reference_space`.

**Note:** No `singlecellreconstruction` table exists yet in the combined datasets. This notebook (or a successor) would be the first to create it.


## 1 — Axon morphology features

`AxonRawReatureWide.csv` → `CellFeatureDefinition` + `CellFeatureSet` + `cellfeatures/wnm_exc_axon_features`

In [21]:
# Build CellFeatureDefinition objects from column names (all float64 → '<f8')
_axon_feat_cols = [c for c in _wnm_axon.columns if c != "specimen_id"]

axon_ft_defs = [
    CellFeatureDefinition(id=col, data_type="<f8")
    for col in _axon_feat_cols
]

print(f"Defined {len(axon_ft_defs)} CellFeatureDefinitions")


Defined 51 CellFeatureDefinitions


In [22]:
# Write CellFeatureDefinition rows
FEATDEF_PATH = os.path.join(COMBINED_ROOT, "cellfeaturedefinition/wnm_exc_axon")

schema = build_arrow_schema(CellFeatureDefinition)
table = models_to_table(axon_ft_defs, schema)
table = attach_linkml_metadata(table, linkml_class="CellFeatureDefinition")

write_deltalake(FEATDEF_PATH, table, mode="overwrite")
print(f"Wrote {len(axon_ft_defs)} CellFeatureDefinition rows → {FEATDEF_PATH}")


Wrote 51 CellFeatureDefinition rows → /scratch/combined_datasets/cellfeaturedefinition/wnm_exc_axon


In [23]:
# Create and write CellFeatureSet
axon_fset = CellFeatureSet(
    id="wnm_exc_axon_features",
    description=(
        "Axon and apical dendrite morphology features for WNM excitatory neurons "
        "(visp_exc_wnm dataset). Includes axon geometry PCs, extent, branching, "
        "apical dendrite metrics, and soma surface area."
    ),
    feature_definition_ids=[fd.id for fd in axon_ft_defs],
    extraction_method="Computed via AllenInstitute skeleton_keys; doi.org/10.1101/2023.11.25.568393",
)

FEATSET_PATH = os.path.join(COMBINED_ROOT, "cellfeatureset/wnm_exc_axon")

schema = build_arrow_schema(CellFeatureSet)
table = models_to_table([axon_fset], schema)
table = attach_linkml_metadata(table, linkml_class="CellFeatureSet")

write_deltalake(FEATSET_PATH, table, mode="overwrite")
print(f"Wrote CellFeatureSet '{axon_fset.id}' → {FEATSET_PATH}")


Wrote CellFeatureSet 'wnm_exc_axon_features' → /scratch/combined_datasets/cellfeatureset/wnm_exc_axon


In [24]:
# Build wide-form feature matrix and write to cellfeatures/wnm_exc_axon_features
# Filter to the 341-cell intersection (all_cell_ids) to match DataItem table
AXON_FEAT_PATH = os.path.join(COMBINED_ROOT, "cellfeatures/wnm_exc_axon_features")

schema = build_cell_feature_matrix_schema(axon_fset, axon_ft_defs, cell_index_column="id")

df = _wnm_axon.copy()
df["id"] = df["specimen_id"].astype(str)
df = df.drop("specimen_id", axis=1)
df = df[df["id"].isin(all_cell_ids)].reset_index(drop=True)  # drop 4 cells not in meta
df["project_id"] = "visp_wnm"
df["feature_set_id"] = "wnm_exc_axon_features"

for fd in axon_ft_defs:
    df[fd.id] = df[fd.id].astype(np.dtype(fd.data_type))

assert len(df) == len(all_cell_ids), f"Expected {len(all_cell_ids)} rows, got {len(df)}"

table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
write_deltalake(AXON_FEAT_PATH, table, mode="overwrite", partition_by=["project_id", "feature_set_id"])
print(f"Wrote {len(df)} rows × {len(_axon_feat_cols)} features → {AXON_FEAT_PATH}")


Wrote 341 rows × 51 features → /scratch/combined_datasets/cellfeatures/wnm_exc_axon_features


In [25]:
# Verification
df_check = pl.read_delta(AXON_FEAT_PATH)
print(f"Shape: {df_check.shape}")
df_check.head(3)


Shape: (341, 54)


id,apical_dendrite_bias_x,apical_dendrite_bias_y,apical_dendrite_depth_pc_0,apical_dendrite_depth_pc_1,apical_dendrite_depth_pc_2,apical_dendrite_depth_pc_3,apical_dendrite_depth_pc_4,apical_dendrite_early_branch_path,apical_dendrite_extent_x,apical_dendrite_extent_y,apical_dendrite_frac_above_axon,apical_dendrite_frac_below_axon,apical_dendrite_frac_intersect_axon,apical_dendrite_max_branch_order,apical_dendrite_max_euclidean_distance,apical_dendrite_max_path_distance,apical_dendrite_mean_contraction,apical_dendrite_mean_diameter,apical_dendrite_mean_moments_along_max_distance_projection,apical_dendrite_num_branches,apical_dendrite_num_outer_bifurcations,apical_dendrite_soma_percentile_x,apical_dendrite_soma_percentile_y,apical_dendrite_std_moments_along_max_distance_projection,apical_dendrite_total_length,apical_dendrite_total_surface_area,axon_bias_x,axon_bias_y,axon_depth_pc_0,axon_depth_pc_1,axon_depth_pc_2,axon_depth_pc_3,axon_depth_pc_4,axon_emd_with_apical_dendrite,axon_exit_distance,axon_exit_theta,axon_extent_x,axon_extent_y,axon_frac_above_apical_dendrite,axon_frac_below_apical_dendrite,axon_frac_intersect_apical_dendrite,axon_max_branch_order,axon_max_euclidean_distance,axon_max_path_distance,axon_mean_contraction,axon_num_branches,axon_soma_percentile_x,axon_soma_percentile_y,axon_total_length,soma_aligned_dist_from_pia,soma_surface_area,project_id,feature_set_id
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""17109_6201-X4328-Y6753_reg""",21.311538,2.306327,-446.959647,-77.960333,-116.411504,-89.641313,-76.897975,0.479785,291.778779,240.142742,0.0,0.0,1.0,5.0,290.537115,320.255996,0.918293,2.0,-0.045459,31.0,0.0,0.477666,0.311239,0.126436,1907.769579,11986.869786,3.040686,487.949889,-118.018264,64.721192,878.933487,206.633126,871.467523,36.500205,0.0,0.296363,1009.404761,879.18749,0.484909,0.036059,0.479033,18.0,782.489366,2295.547989,0.775924,133.0,0.456671,0.2913,32679.601463,755.648634,0.0,"""visp_wnm""","""wnm_exc_axon_features"""
"""17109_6301-X4756-Y24516_reg""",39.591795,17.363311,-456.398402,-70.969916,-123.361715,-92.973863,-99.533545,0.484623,255.485239,181.024288,0.0,0.0,1.0,5.0,276.476541,309.965281,0.95142,2.0,0.035383,42.0,0.0,0.472446,0.456966,0.129964,2221.638776,13958.968115,15.367443,490.193871,-277.75066,25.648102,571.147506,62.002695,621.814377,39.566076,0.0,0.397055,1001.485259,989.55409,0.501198,0.071934,0.426868,11.0,798.003108,1794.220795,0.789746,83.0,0.434223,0.19318,24944.239274,779.803826,0.0,"""visp_wnm""","""wnm_exc_axon_features"""
"""17109_6601-X4384-Y7436_reg""",110.472066,-7.655321,-447.486796,-103.720271,-125.56466,-117.194281,-51.706742,0.49886,331.019359,204.938937,0.0,0.0,1.0,5.0,265.268786,324.289849,0.925244,1.863124,0.078128,47.0,0.477121,0.376812,0.468062,0.181604,2541.558556,14893.488672,28.778128,212.211203,-665.616843,305.037634,644.22927,-52.224096,751.065146,16.076402,0.0,0.266886,990.430541,819.257725,0.216496,0.135099,0.648406,10.0,722.175168,2298.330681,0.773319,79.0,0.297082,0.511213,20598.298943,727.831495,0.0,"""visp_wnm""","""wnm_exc_axon_features"""


## 2 — Projection matrices (ipsi / contra)

`ProjectionMatrix_tip_and_branch_roll_up.csv` → two wide-form delta tables + one `ProjectionMeasurementMatrix` metadata table.

Column prefixes `ipsi_` / `contra_` are stripped; region acronyms become column names.

In [37]:
# Split columns by laterality and normalise cell IDs
_ipsi_cols   = [c for c in _wnm_proj.columns if c.startswith("ipsi_")]
_contra_cols = [c for c in _wnm_proj.columns if c.startswith("contra_")]

_ipsi_regions   = [c[len("ipsi_"):]   for c in _ipsi_cols]
_contra_regions = [c[len("contra_"):] for c in _contra_cols]

_proj_cell_ids = [i.replace(".swc", "") for i in _wnm_proj.index]

print(f"ipsi regions  : {len(_ipsi_regions)}")
print(f"contra regions: {len(_contra_regions)}")
print(f"cells         : {len(_proj_cell_ids)}")


ipsi regions  : 152
contra regions: 68
cells         : 345


In [38]:
# Create ProjectionMeasurementMatrix metadata objects (one per laterality)
# data_item_index uses the 341-cell intersection (all_cell_ids)
PROJ_META_PATH = os.path.join(COMBINED_ROOT, "projectionmeasurementmatrix")

ipsi_pmm = ProjectionMeasurementMatrix(
    id="wnm_exc_proj_ipsi",
    description=(
        "Ipsilateral axon projection matrix for WNM excitatory neurons (visp_exc_wnm). "
        "Tip and branch roll-up; 341 cells × 152 brain regions."
    ),
    measurement_type=ProjectionMeasurementType.NUMBER_OF_TIPS.value,
    modality=Modality.MORPHOLOGY.value,
    region_index=_ipsi_regions,
    data_item_index=all_cell_ids,
    values=os.path.join(COMBINED_ROOT, "projectionmeasurementmatrix/wnm_exc_proj_ipsi"),
)

contra_pmm = ProjectionMeasurementMatrix(
    id="wnm_exc_proj_contra",
    description=(
        "Contralateral axon projection matrix for WNM excitatory neurons (visp_exc_wnm). "
        "Tip and branch roll-up; 341 cells × 68 brain regions."
    ),
    measurement_type=ProjectionMeasurementType.NUMBER_OF_TIPS.value,
    modality=Modality.MORPHOLOGY.value,
    region_index=_contra_regions,
    data_item_index=all_cell_ids,
    values=os.path.join(COMBINED_ROOT, "projectionmeasurementmatrix/wnm_exc_proj_contra"),
)

schema = build_arrow_schema(ProjectionMeasurementMatrix)
table = models_to_table([ipsi_pmm, contra_pmm], schema)
table = attach_linkml_metadata(table, linkml_class="ProjectionMeasurementMatrix")

write_deltalake(PROJ_META_PATH, table, mode="overwrite")
print(f"Wrote 2 ProjectionMeasurementMatrix metadata rows → {PROJ_META_PATH}")


Wrote 2 ProjectionMeasurementMatrix metadata rows → /scratch/combined_datasets/projectionmeasurementmatrix


In [39]:
# Write ipsi wide-form matrix — filtered to 341-cell intersection
PROJ_IPSI_PATH = os.path.join(COMBINED_ROOT, "projectionmeasurementmatrix/wnm_exc_proj_ipsi")

ipsi_df = _wnm_proj[_ipsi_cols].copy()
ipsi_df.columns = _ipsi_regions
ipsi_df.index = _proj_cell_ids
ipsi_df = ipsi_df.loc[ipsi_df.index.isin(all_cell_ids)]  # drop 4 cells not in meta
ipsi_df = ipsi_df.reset_index().rename(columns={"index": "id"})
ipsi_df["project_id"] = "visp_wnm"
ipsi_df["dataset_id"] = "visp_exc_wnm"

assert len(ipsi_df) == len(all_cell_ids), f"Expected {len(all_cell_ids)} rows, got {len(ipsi_df)}"

write_deltalake(PROJ_IPSI_PATH, pa.Table.from_pandas(ipsi_df, preserve_index=False),
                mode="overwrite", partition_by=["project_id"])
print(f"Wrote {ipsi_df.shape} → {PROJ_IPSI_PATH}")


Wrote (341, 155) → /scratch/combined_datasets/projectionmeasurementmatrix/wnm_exc_proj_ipsi


In [40]:
# Write contra wide-form matrix — filtered to 341-cell intersection
PROJ_CONTRA_PATH = os.path.join(COMBINED_ROOT, "projectionmeasurementmatrix/wnm_exc_proj_contra")

contra_df = _wnm_proj[_contra_cols].copy()
contra_df.columns = _contra_regions
contra_df.index = _proj_cell_ids
contra_df = contra_df.loc[contra_df.index.isin(all_cell_ids)]  # drop 4 cells not in meta
contra_df = contra_df.reset_index().rename(columns={"index": "id"})
contra_df["project_id"] = "visp_wnm"
contra_df["dataset_id"] = "visp_exc_wnm"

assert len(contra_df) == len(all_cell_ids), f"Expected {len(all_cell_ids)} rows, got {len(contra_df)}"

write_deltalake(PROJ_CONTRA_PATH, pa.Table.from_pandas(contra_df, preserve_index=False),
                mode="overwrite", partition_by=["project_id"])
print(f"Wrote {contra_df.shape} → {PROJ_CONTRA_PATH}")


Wrote (341, 71) → /scratch/combined_datasets/projectionmeasurementmatrix/wnm_exc_proj_contra


In [41]:
# Verification
print("=== projectionmeasurementmatrix (metadata) ===")
display(pl.read_delta(PROJ_META_PATH).select(["id", "measurement_type", "modality"]).to_pandas())

print("\n=== wnm_exc_proj_ipsi (head, first 5 region cols) ===")
df_i = pl.read_delta(PROJ_IPSI_PATH)
print(df_i.shape)
display(df_i.select(["id", "project_id"] + _ipsi_regions[:5]).head(3).to_pandas())


=== projectionmeasurementmatrix (metadata) ===


,id,measurement_type,modality
0,wnm_exc_proj_ipsi,NUMBER_OF_TIPS,MORPHOLOGY
1,wnm_exc_proj_contra,NUMBER_OF_TIPS,MORPHOLOGY



=== wnm_exc_proj_ipsi (head, first 5 region cols) ===
(341, 155)


,id,project_id,VISam,VISp,VISpm,VISrl,CP
0,18864_6734-X4899-Y27447_reg,visp_wnm,8287.70664,34450.175934,483.223644,5737.760785,0.000000
1,191812_7938-X6892-Y25312_reg,visp_wnm,0.00000,794.102517,0.000000,0.000000,9243.339922
2,211550_7718-X19461-Y16950_reg,visp_wnm,0.00000,6473.751624,0.000000,0.000000,0.000000


## 3 — Single cell reconstruction (soma location)

`FullMorphMetaData_Master.csv` → `singlecellreconstruction` table.

Populates `soma_location` (CCF x/y/z) for the 341 cells present in the metadata file.  
`ccf_registered_file` is left empty — storage path not available in this context.

In [31]:
reconstructions = []
for idx, row in _wnm_meta.iterrows():
    cell_id = idx.replace(".swc", "")
    soma_loc = SpatialLocation(
        x=float(row["ccf_soma_x"]),
        y=float(row["ccf_soma_y"]),
        z=float(row["ccf_soma_z"]),
        reference_space="CCF_v3",
    )
    reconstructions.append(SingleCellReconstruction(id=cell_id, soma_location=soma_loc))

print(f"Created {len(reconstructions)} SingleCellReconstruction objects")


Created 341 SingleCellReconstruction objects


In [35]:
SCR_PATH = os.path.join(COMBINED_ROOT, "singlecellreconstruction")

schema = build_arrow_schema(SingleCellReconstruction)
table = models_to_table(reconstructions, schema)
table = attach_linkml_metadata(table, linkml_class="SingleCellReconstruction")

write_deltalake(SCR_PATH, table, mode="overwrite")
print(f"Wrote {len(reconstructions)} SingleCellReconstruction rows → {SCR_PATH}")


Wrote 341 SingleCellReconstruction rows → /scratch/combined_datasets/singlecellreconstruction


In [36]:
# Verification
df_scr = pl.read_delta(SCR_PATH)
print(f"Shape: {df_scr.shape}")
df_scr.head(3)


Shape: (341, 3)


id,ccf_registered_file,soma_location
str,str,str
"""182709_6984-X2452-Y12423_reg""",null,"""{'x': 8899.823, 'y': 643.262, …"
"""182709_7126-X2913-Y10535_reg""",null,"""{'x': 9117.453, 'y': 1064.075,…"
"""182724_5937-X3804-Y11955_reg""",null,"""{'x': 7168.2890000000025, 'y':…"
